# Transformation des données avec PySpark

Ce notebook effectue :
1. Chargement des données depuis HDFS
2. Renommage des colonnes (Anglais -> Français)
3. Traduction des valeurs catégorielles
4. Suppression des doublons et valeurs nulles
5. Sauvegarde sur HDFS

## 1. Configuration Spark et chargement depuis HDFS

In [12]:
import os
os.environ['JAVA_HOME'] = '/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home'

from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

port = "9000"
hdfs_path = f"hdfs://localhost:{port}/user/roissath/data"

spark = SparkSession.builder \
    .appName("transformation_student_performance") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.extraJavaOptions", "--add-opens=java.base/javax.security.auth=ALL-UNNAMED") \
    .getOrCreate()

print(f"Session Spark créée : {spark.version}")

df = spark.read.csv(f"{hdfs_path}/StudentPerformanceFactors.csv", header=True, inferSchema=True)

print(f"Nombre de lignes chargées: {df.count()}")
df.printSchema()

Session Spark créée : 4.0.4
Nombre de lignes chargées: 6607
root
 |-- Hours_Studied: integer (nullable = true)
 |-- Attendance: integer (nullable = true)
 |-- Parental_Involvement: string (nullable = true)
 |-- Access_to_Resources: string (nullable = true)
 |-- Extracurricular_Activities: string (nullable = true)
 |-- Sleep_Hours: integer (nullable = true)
 |-- Previous_Scores: integer (nullable = true)
 |-- Motivation_Level: string (nullable = true)
 |-- Internet_Access: string (nullable = true)
 |-- Tutoring_Sessions: integer (nullable = true)
 |-- Family_Income: string (nullable = true)
 |-- Teacher_Quality: string (nullable = true)
 |-- School_Type: string (nullable = true)
 |-- Peer_Influence: string (nullable = true)
 |-- Physical_Activity: integer (nullable = true)
 |-- Learning_Disabilities: string (nullable = true)
 |-- Parental_Education_Level: string (nullable = true)
 |-- Distance_from_Home: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Exam_Score: int

## 2. Mapping des colonnes et valeurs

In [13]:
# Mapping des noms de colonnes (Anglais -> Français)
COLUMN_MAPPING = {
    "Hours_Studied": "Heures_Etudiees",
    "Attendance": "Presence",
    "Parental_Involvement": "Implication_Parentale",
    "Access_to_Resources": "Acces_aux_Ressources",
    "Extracurricular_Activities": "Activites_Extrascolaires",
    "Sleep_Hours": "Heures_Sommeil",
    "Previous_Scores": "Scores_Precedents",
    "Motivation_Level": "Niveau_Motivation",
    "Internet_Access": "Acces_Internet",
    "Tutoring_Sessions": "Sessions_Tutorat",
    "Family_Income": "Revenu_Famille",
    "Teacher_Quality": "Qualite_Enseignant",
    "School_Type": "Type_Ecole",
    "Peer_Influence": "Influence_Entourage",
    "Physical_Activity": "Activite_Physique",
    "Learning_Disabilities": "Troubles_Apprentissage",
    "Parental_Education_Level": "Niveau_Education_Parents",
    "Distance_from_Home": "Distance_Maison",
    "Gender": "Genre",
    "Exam_Score": "Score_Examen"
}

# Mapping des valeurs (Anglais -> Français)
VALUE_MAPPING = {
    # Low/Medium/High
    "Low": "Bas",
    "Medium": "Moyen",
    "High": "Haut",
    # Yes/No
    "Yes": "Oui",
    "No": "Non",
    # School Type
    "Public": "Publique",
    "Private": "Privee",
    # Peer Influence
    "Negative": "Negative",
    "Neutral": "Neutre",
    "Positive": "Positif",
    # Parental Education Level
    "High School": "Lycee",
    "College": "Licence",
    "Postgraduate": "Master",
    # Distance from Home
    "Near": "Proche",
    "Moderate": "Moderee",
    "Far": "Loin",
    # Gender
    "Male": "Homme",
    "Female": "Femme"
}

print("Mappings définis avec succès")

Mappings définis avec succès


## 3. Renommage des colonnes

In [14]:
# Renommer les colonnes
for old_name, new_name in COLUMN_MAPPING.items():
    if old_name in df.columns:
        df = df.withColumnRenamed(old_name, new_name)

print("Colonnes renommées:")
df.printSchema()

Colonnes renommées:
root
 |-- Heures_Etudiees: integer (nullable = true)
 |-- Presence: integer (nullable = true)
 |-- Implication_Parentale: string (nullable = true)
 |-- Acces_aux_Ressources: string (nullable = true)
 |-- Activites_Extrascolaires: string (nullable = true)
 |-- Heures_Sommeil: integer (nullable = true)
 |-- Scores_Precedents: integer (nullable = true)
 |-- Niveau_Motivation: string (nullable = true)
 |-- Acces_Internet: string (nullable = true)
 |-- Sessions_Tutorat: integer (nullable = true)
 |-- Revenu_Famille: string (nullable = true)
 |-- Qualite_Enseignant: string (nullable = true)
 |-- Type_Ecole: string (nullable = true)
 |-- Influence_Entourage: string (nullable = true)
 |-- Activite_Physique: integer (nullable = true)
 |-- Troubles_Apprentissage: string (nullable = true)
 |-- Niveau_Education_Parents: string (nullable = true)
 |-- Distance_Maison: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Score_Examen: integer (nullable = true)



## 4. Traduction des valeurs catégorielles

In [15]:
# Colonnes avec valeurs Low/Medium/High
cols_low_medium_high = [
    "Implication_Parentale", "Acces_aux_Ressources", "Niveau_Motivation",
    "Revenu_Famille", "Qualite_Enseignant"
]

for c in cols_low_medium_high:
    if c in df.columns:
        df = df.withColumn(c,
            when(col(c) == "Low", "Bas")
            .when(col(c) == "Medium", "Moyen")
            .when(col(c) == "High", "Haut")
            .otherwise(col(c))
        )

# Colonnes avec valeurs Yes/No
cols_yes_no = ["Activites_Extrascolaires", "Acces_Internet", "Troubles_Apprentissage"]

for c in cols_yes_no:
    if c in df.columns:
        df = df.withColumn(c,
            when(col(c) == "Yes", "Oui")
            .when(col(c) == "No", "Non")
            .otherwise(col(c))
        )

# Type d'école
if "Type_Ecole" in df.columns:
    df = df.withColumn("Type_Ecole",
        when(col("Type_Ecole") == "Public", "Publique")
        .when(col("Type_Ecole") == "Private", "Privee")
        .otherwise(col("Type_Ecole"))
    )

# Influence de l'entourage
if "Influence_Entourage" in df.columns:
    df = df.withColumn("Influence_Entourage",
        when(col("Influence_Entourage") == "Negative", "Negative")
        .when(col("Influence_Entourage") == "Neutral", "Neutre")
        .when(col("Influence_Entourage") == "Positive", "Positif")
        .otherwise(col("Influence_Entourage"))
    )

# Niveau d'éducation des parents
if "Niveau_Education_Parents" in df.columns:
    df = df.withColumn("Niveau_Education_Parents",
        when(col("Niveau_Education_Parents") == "High School", "Lycee")
        .when(col("Niveau_Education_Parents") == "College", "Licence")
        .when(col("Niveau_Education_Parents") == "Postgraduate", "Master")
        .otherwise(col("Niveau_Education_Parents"))
    )

# Distance de la maison
if "Distance_Maison" in df.columns:
    df = df.withColumn("Distance_Maison",
        when(col("Distance_Maison") == "Near", "Proche")
        .when(col("Distance_Maison") == "Moderate", "Moderee")
        .when(col("Distance_Maison") == "Far", "Loin")
        .otherwise(col("Distance_Maison"))
    )

# Genre
if "Genre" in df.columns:
    df = df.withColumn("Genre",
        when(col("Genre") == "Male", "Homme")
        .when(col("Genre") == "Female", "Femme")
        .otherwise(col("Genre"))
    )

print("Valeurs traduites avec succès")
df.show(5)

Valeurs traduites avec succès
+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+----------------------+------------------------+---------------+-----+------------+
|Heures_Etudiees|Presence|Implication_Parentale|Acces_aux_Ressources|Activites_Extrascolaires|Heures_Sommeil|Scores_Precedents|Niveau_Motivation|Acces_Internet|Sessions_Tutorat|Revenu_Famille|Qualite_Enseignant|Type_Ecole|Influence_Entourage|Activite_Physique|Troubles_Apprentissage|Niveau_Education_Parents|Distance_Maison|Genre|Score_Examen|
+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+----------------------

## 5. Nettoyage : suppression des doublons et valeurs nulles

In [16]:
count_before = df.count()
print(f"Lignes avant nettoyage: {count_before}")

# Supprimer les lignes avec des valeurs nulles
df_clean = df.dropna()
count_after_null = df_clean.count()
print(f"Lignes avec valeurs nulles supprimées: {count_before - count_after_null}")

# Supprimer les doublons
df_clean = df_clean.dropDuplicates()
count_final = df_clean.count()
print(f"Doublons supprimés: {count_after_null - count_final}")

print(f"\nLignes finales: {count_final}")

Lignes avant nettoyage: 6607
Lignes avec valeurs nulles supprimées: 229
Doublons supprimés: 0

Lignes finales: 6378
Doublons supprimés: 0

Lignes finales: 6378


## 6. Aperçu des données transformées

In [17]:
df_clean.show(10)
df_clean.printSchema()

+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+----------------------+------------------------+---------------+-----+------------+
|Heures_Etudiees|Presence|Implication_Parentale|Acces_aux_Ressources|Activites_Extrascolaires|Heures_Sommeil|Scores_Precedents|Niveau_Motivation|Acces_Internet|Sessions_Tutorat|Revenu_Famille|Qualite_Enseignant|Type_Ecole|Influence_Entourage|Activite_Physique|Troubles_Apprentissage|Niveau_Education_Parents|Distance_Maison|Genre|Score_Examen|
+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+----------------------+------------------------+----

## 7. Sauvegarde sur HDFS

In [18]:
# Sauvegarder le fichier transformé sur HDFS
output_path = f"{hdfs_path}/StudentPerformanceFactors_propre.csv"

# Écrire en CSV avec header (en un seul fichier avec coalesce)
df_clean.coalesce(1).write.mode("overwrite").option("header", True).csv(output_path)

print(f"Fichier sauvegardé sur HDFS: {output_path}")

Fichier sauvegardé sur HDFS: hdfs://localhost:9000/user/roissath/data/StudentPerformanceFactors_propre.csv


In [19]:
# Vérification : relire le fichier sauvegardé
df_verif = spark.read.csv(output_path, header=True, inferSchema=True)
print(f"Vérification - Lignes lues: {df_verif.count()}")
df_verif.show(5)

Vérification - Lignes lues: 6378
+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+----------------------+------------------------+---------------+-----+------------+
|Heures_Etudiees|Presence|Implication_Parentale|Acces_aux_Ressources|Activites_Extrascolaires|Heures_Sommeil|Scores_Precedents|Niveau_Motivation|Acces_Internet|Sessions_Tutorat|Revenu_Famille|Qualite_Enseignant|Type_Ecole|Influence_Entourage|Activite_Physique|Troubles_Apprentissage|Niveau_Education_Parents|Distance_Maison|Genre|Score_Examen|
+---------------+--------+---------------------+--------------------+------------------------+--------------+-----------------+-----------------+--------------+----------------+--------------+------------------+----------+-------------------+-----------------+-------------------

In [20]:
# Fermer la session Spark
spark.stop()
print("Session Spark fermée")

Session Spark fermée
